Introduction

This project focuses on network intrusion detection using the NSL-KDD dataset, an enhanced version of the well-known KDD'99 dataset.

The goal is to develop several models capable of distinguishing normal traffic from attacks by analyzing various network parameters.
My colleagues first imported, cleaned, and preprocessed the data (encoding, normalization, and balancing with SMOTE).
An exploratory analysis allowed us to understand the dataset's structure and identify key variables.

Decision Arbre

In [ ]:
# Import
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Création et entraînement du modèle
dt_model = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=10,
    random_state=42
)

dt_model.fit(X_train_bal, y_train_bal)

# Prédictions sur le jeu de test
y_pred_dt = dt_model.predict(X_test_scaled)

# Évaluation des performances
print("Decision Tree Classifier Report:")
print(classification_report(y_test, y_pred_dt))

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred_dt, labels=['normal', 'anomaly'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['normal', 'anomaly'])
disp.plot(cmap='Blues')
plt.title("Decision Tree - Confusion Matrix")
plt.show()

# Importance des caractéristiques
importances = pd.Series(dt_model.feature_importances_, index=X_train_bal.columns)
importances = importances.sort_values(ascending=False)[:10]  # top 10 features

plt.figure(figsize=(8,5))
importances.plot(kind='barh', color='teal')
plt.title("Top 10 Feature Importances (Decision Tree)")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.gca().invert_yaxis()
plt.show()

Objective

After logistic regression, the objective was to test a more flexible and visual model such as a decision tree.

This model allows us to understand how certain variables directly influence intrusion detection, while also capturing more complex relationships between data points.

Method

The model was trained on the balanced dataset using SMOTE, with entropy as the criterion for measuring the quality of separations.
The maximum depth was limited to 10 levels to avoid overfitting.
Evaluation was performed on the normalized test set, with the same metrics as before: accuracy, precision, recall, F1 score, and confusion matrix.

Results

The model achieves approximately 75% overall accuracy, results close to those obtained with logistic regression.

It accurately identifies normal connections but still misses some attacks.

Anomaly Accuracy: 0.87
Anomaly Recall: 0.67
Overall F1 Score: 0.75

The confusion matrix shows that:
8,567 attacks were correctly detected,
 4,266 were classified as normal,
and 1,326 normal connections were incorrectly classified as abnormal.


The decision tree offers a good compromise between interpretation and performance.
It allows visualization of the criteria leading to detection, but its accuracy remains limited.
This step shows that a more robust model, such as a Random Forest, could improve results while reducing classification errors.


Random Forest

In [ ]:

from sklearn.ensemble import RandomForestClassifier


rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)


rf_model.fit(X_train_bal, y_train_bal)


y_pred_rf = rf_model.predict(X_test_scaled)


print("Random Forest Classifier Report:")
print(classification_report(y_test, y_pred_rf))


cm = confusion_matrix(y_test, y_pred_rf, labels=['normal', 'anomaly'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['normal', 'anomaly'])
disp.plot()
plt.title("Random Forest - Confusion Matrix")
plt.show()


importances = pd.Series(rf_model.feature_importances_, index=X_train_bal.columns).sort_values(ascending=False)[:15]
plt.figure(figsize=(8,5))
importances.plot(kind='barh')
plt.gca().invert_yaxis()
plt.title("Top 15 Feature Importances (Random Forest)")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

Objective

After the decision tree model, the objective was to improve the model's performance and stability by using an ensemble approach such as the Random Forest Classifier.

This model combines several decision trees trained on different subsamples of the dataset to achieve better generalization and reduce overfitting.


Methodology

The model was trained on the balanced training set (after SMOTE), with the following key parameters:
- n_estimators = 100 (number of trees),
- criterion = 'entropy',
- max_depth = 10,
- random_state = 42.

The evaluation was performed on the normalized test set using the same metrics: precision, recall, F1 score, accuracy, and confusion matrix.
The importance of the variables was also calculated to understand the most decisive criteria in intrusion detection.

Résults

Exactitude (Accuracy)	≈ 0.77
Précision moyenne	≈ 0.80
Rappel moyen	≈ 0.79
F1-score moyen	≈ 0.77

Details by class:
- "Anomaly" class: Accuracy = 0.89, Recall = 0.69, F1 = 0.77
- "Normal" class: Accuracy = 0.68, Recall = 0.89, F1 = 0.77

The model slightly improves results compared to the decision tree, particularly in terms of overall stability.

It better balances detection between normal connections and attacks, while reducing the risk of overfitting.

Confusion matrix:
- 8,792 anomalies correctly detected
- 4,041 anomalies classified as normal
- 8,625 normal connections correctly identified
- 1,086 normal connections classified as anomalies

These figures show that the model reduces errors compared to the decision tree, especially for the "normal" class.

Random Forest slightly outperforms decision trees in terms of accuracy and recall, while remaining robust against overfitting.
Its ability to combine multiple trees allows for greater stability and more reliable intrusion detection.

However, the model remains relatively resource-intensive to run, and it can still be improved by refining the parameters or testing more efficient approaches such as SVM.

SVM

In [ ]:

from sklearn.svm import SVC


svm_model = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    class_weight='balanced',
    random_state=42
)

svm_model.fit(X_train_bal, y_train_bal)


y_pred_svm = svm_model.predict(X_test_scaled)


print(" SVM Classifier Report:")
print(classification_report(y_test, y_pred_svm))


cm = confusion_matrix(y_test, y_pred_svm, labels=['normal', 'anomaly'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['normal', 'anomaly'])
disp.plot(cmap='Blues')
plt.title("SVM - Confusion Matrix")
plt.show()

Objective

Following the evaluation of tree models (Decision Tree and Random Forest), the objective was to assess a maximum margin model, the Support Vector Machine (SVM).

The aim is to verify whether a more theoretical model based on optimal class boundaries can offer better separation between normal traffic and attacks, while limiting overfitting.

Method

The SVM model was trained on the preprocessed dataset (encoded, standardized and balanced with SMOTE).

Main parameters:
- kernel = 'rbf' (Gaussian kernel, suitable for non-linear data),
- C = 1.0 (default regularization parameter),
- gamma = 'scale'.

The evaluation was performed on the full test set with the same performance indicators as before.

Results :

Exactitude (Accuracy) ≈ 0.81
Précision moyenne  ≈ 0.82
Rappel moyen ≈ 0.80
F1-score moyen ≈ 0.81

Details by class:
- Class “anomaly”: Precision = 0.88, Recall = 0.77, F1 = 0.82
- Class “normal”: Precision = 0.75, Recall = 0.86, F1 = 0.80

The model slightly improves overall performance compared to Random Forest, particularly in the recall of the “normal” class.
It demonstrates better generalization capabilities and a more stable balance between precision and recall.

The SVM offers the best overall performance among the tested models, with an average accuracy and F1 score around 0.81.
Its nonlinear kernel allows it to capture complex relationships between variables, resulting in more precise intrusion detection.

However, computation time remains its main drawback, especially with large datasets.

Apply cross-validation to check overfitting

In [ ]:
#Import
from sklearn.model_selection import cross_val_score

# 5-fold cross-validation
print(" Validation croisée (5 folds) sur le modèle SVM...")



cv_scores = cross_val_score(
    svm_model,
    X_train_bal,
    y_train_bal,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

print("\n Cross-validation results:")
print("Score by fold :", np.round(cv_scores, 4))
print(" Average :", round(cv_scores.mean(), 4))
print("Typical gap :", round(cv_scores.std(), 4))

Cross-validation

Cross-validation allows you to verify the reliability and stability of a model.

The principle is to divide the training set into several parts (for example, 5). The model is trained on one part of the data and tested on another, then this process is repeated several times to obtain an average performance.
This method ensures that the model does not overfit and that it generalizes well to new data.